# Annotated Bibliography

Render a formatted annotated bibliography from the paper database.

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from IPython.display import display, Markdown

from src.storage.paper_store import PaperStore
from src.analysis.summarizer import generate_summary

store = PaperStore()
papers = store.get_all()
print(f"Loaded {len(papers)} papers")

## Full Summary Report

In [ ]:
report = generate_summary(store)
display(Markdown(report))

## Detailed View: High-Relevance Papers

In [ ]:
# Show papers marked as read or annotated with full details
reviewed = [p for p in papers if p.read_status in ("read", "annotated")]
reviewed.sort(key=lambda p: (-(p.relevance_score or 0), p.title))

if not reviewed:
    display(Markdown("*No papers have been reviewed yet. Mark papers as 'read' or 'annotated' in the search notebook.*"))
else:
    for p in reviewed:
        md = f"### {p.title}\n\n"
        md += f"**Authors:** {', '.join(p.authors)}\n\n"
        md += f"**Year:** {p.year}  \n"
        if p.citation_count:
            md += f"**Citations:** {p.citation_count}  \n"
        if p.relevance_score:
            md += f"**Relevance:** {p.relevance_score:.1f}/1.0  \n"
        md += f"**Tags:** {', '.join(p.tags)}  \n\n"
        if p.tldr:
            md += f"**TLDR:** {p.tldr}\n\n"
        if p.abstract:
            md += f"**Abstract:** {p.abstract}\n\n"
        if p.notes:
            md += f"**Notes:** {p.notes}\n\n"
        md += "---\n"
        display(Markdown(md))

## Export BibTeX

In [ ]:
from src.storage.bibtex_export import export_to_bibtex

bib_path = project_root / "data" / "bibliography.bib"
bibtex = export_to_bibtex(store, output_path=bib_path)
print(f"Exported {len(papers)} papers to {bib_path}")
print(f"\nFirst 500 characters:\n")
print(bibtex[:500])